In [1]:
import pandas as pd
import csv
import os

In [ ]:
def strip_object_columns(df: pd.DataFrame) -> pd.DataFrame:

    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str).str.strip()
    
    return df

In [ ]:
def clean_numeric_columns(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame: # adds a threshold of 0.7 (70%); so unless the row has 70% >= non-null values that are numeric, it will execute, if not, no conversion will be done on that row 
    df = df.copy()
    
    for col in df.columns:
        if col.lower() in {}:
            continue
        
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric = pd.to_numeric(cleaned, errors="coerce") # updated "ignore" to "coerce" due to FutureWarning

        non_null = cleaned.notna().sum()
        numeric_count = numeric.notna().sum()

        if non_null > 0 and numeric_count / non_null >= threshold: 
            df[col] = numeric
    
    return df

In [ ]:
def remove_empty_rows(df: pd.DataFrame) -> pd.DataFrame: # this will remove the rows where the numeric columns are NaN
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    if len(numeric_cols) == 0:
        return df # this checks the number of numeric rows; if 0, then it will end
    
    return df.dropna(subset=numeric_cols, how='all')

In [ ]:
def format_numbers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    def format_whole_numbers(x):
        if pd.isna(x):
            return x
        if isinstance(x, float) and x.is_integer():
            return int(x)
        return x

    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)

    return df

# function that formats numbers with no decimal value to whole integers and leaves float values untouched

In [ ]:
def add_suffix_cleaned(file_name, suffix='_cleaned'): # this will add the suffix 'cleaned' to the file name separated by an underscore
    base, ext = file_name.rsplit(".", 1) 
    return f"{base}{suffix}.{ext}" # splits the name of the file into 'base' and 'ext'; adds '_cleaned' to the new file name before the file extension

In [ ]:
def rename_csv_files(input_dir: str, output_dir: str, csv_rename_files: dict): # rename csv files with selective names; no blanket names

    for old_file, new_file in csv_rename_files.items():
        input_path = f"{input_dir}/{old_file}"
        output_path = f"{output_dir}/{new_file}"

        df = pd.read_csv(input_path)
        df.to_csv(output_path, index=False)

        print(f"Renamed {old_file} to {new_file}")

In [ ]:
def batch_clean_csv( # cleans a batch of csv files and adds '_cleaned' before the .ext to show it is finalized
    input_dir: str,
    output_dir: str,
    filenames: list[str]
    ):
    
    for filename in filenames:
        input_file = f"{input_dir}/{filename}"
        output_file = f"{output_dir}/{add_suffix_cleaned(filename)}"

        df = pd.read_csv(input_file)    

        df = strip_object_columns(df)
        df = clean_numeric_columns(df)
        df = remove_empty_rows(df)
        df = format_numbers(df)

        df.to_csv(output_file, index=False)
        print(f"Saved cleaned CSV: {output_file}")

In [ ]:
csv_rename_files = {
    "fod_demo_soc_table_1.csv": "fod_demo_soc.csv",
    "fod_demo_soc_table_m1.csv": "fod_demo_soc_moe.csv",
    "fod_earn_age_table_3.csv": "fod_earn_age.csv",
    "fod_earn_edu_att_table_5-attainment_earnings.csv": "fod_earn_edu_att.csv",
    "fod_earn_metro_table_7.csv": "fod_earn_metro.csv",
    "fod_earn_metro_table_m3.csv": "fod_earn_metro_moe.csv",
    "fod_earn_race_table_4-race_earnings.csv": "fod_earn_race.csv",
    "fod_earn_sex_table_2.csv": "fod_earn_sex.csv",
    "fod_metro_table_6.csv": "fod_metro.csv",
    "fod_metro_table_m2.csv": "fod_metro_moe.csv"
}

input_dir = "../data/education/working"
output_dir = "../data/education/working"

rename_csv_files(input_dir, output_dir, csv_rename_files)

In [ ]:
csv_files = [
    "fobd_by_state.csv",
    "fod_demo_soc.csv",
    "fod_demo_soc_moe.csv",
    "fod_earn_age.csv",
    "fod_earn_edu_att.csv",
    "fod_earn_metro.csv",
    "fod_earn_metro_moe.csv",
    "fod_earn_race.csv",
    "fod_earn_sex.csv",
    "fod_metro.csv"
    ]

input_dir = "../data/food/working"
output_dir = "../data/food/cleaned"

batch_clean_csv(csv_files, input_dir, output_dir)